In [1]:
%pip install opencv-python numpy

   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   ------ --------------------------------- 7.1/44.0 MB 36.4 MB/s eta 0:00:02
   ------------ --------------------------- 14.2/44.0 MB 35.6 MB/s eta 0:00:01
   ------------------- -------------------- 21.2/44.0 MB 34.4 MB/s eta 0:00:01
   ------------------------- -------------- 28.3/44.0 MB 34.5 MB/s eta 0:00:01
   -------------------------------- ------- 35.4/44.0 MB 34.6 MB/s eta 0:00:01
   -------------------------------------- - 42.2/44.0 MB 34.4 MB/s eta 0:00:01
   ---------------------------------------- 44.0/44.0 MB 32.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.4 MB ? eta -:--:--
   --------------------- ------------------ 6.8/12.4 MB 34.9 MB/s eta 0:00:01
   ---------------------------------------- 12.4/12.4 MB 32.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import cv2

# Load image
image = cv2.imread("../sample_patch.png")

# Resize to 512x512
resized = cv2.resize(image, (512, 512), interpolation=cv2.INTER_AREA)

# Save result
cv2.imwrite("../sample_patch_512.png", resized)

True

In [25]:
import cv2
import numpy as np

# -----------------------------
# Configuration
# -----------------------------
IMAGE_PATH = "../sample_patch_512.png"

MASK_OUTPUT = "../mask_512.png"

BRUSH_SIZE = 20

# -----------------------------
# Load image
# -----------------------------
image = cv2.imread(IMAGE_PATH)

if image is None:
    raise FileNotFoundError(f"Cannot find {IMAGE_PATH}")

display = image.copy()

mask = np.zeros(image.shape[:2], dtype=np.uint8)

drawing = False


# -----------------------------
# Mouse callback
# -----------------------------
def draw(event, x, y, flags, param):
    global drawing, display, mask

    if event == cv2.EVENT_LBUTTONDOWN:
        drawing = True

    elif event == cv2.EVENT_MOUSEMOVE and drawing:
        cv2.circle(display, (x, y), BRUSH_SIZE, (0, 0, 255), -1)
        cv2.circle(mask, (x, y), BRUSH_SIZE, 255, -1)

    elif event == cv2.EVENT_LBUTTONUP:
        drawing = False
        cv2.circle(display, (x, y), BRUSH_SIZE, (0, 0, 255), -1)
        cv2.circle(mask, (x, y), BRUSH_SIZE, 255, -1)


# -----------------------------
# Window
# -----------------------------
cv2.namedWindow("Draw Mask")
cv2.setMouseCallback("Draw Mask", draw)

print("Controls")
print("---------------------")
print("Left mouse = draw")
print("c = clear")
print("+ = larger brush")
print("- = smaller brush")
print("s = save mask")
print("ESC = quit")

while True:

    cv2.imshow("Draw Mask", display)

    key = cv2.waitKey(1) & 0xFF

    if key == 27:
        break

    elif key == ord('c'):
        display = image.copy()
        mask[:] = 0

    elif key == ord('+') or key == ord('='):
        BRUSH_SIZE += 2
        print("Brush:", BRUSH_SIZE)

    elif key == ord('-'):
        BRUSH_SIZE = max(1, BRUSH_SIZE - 2)
        print("Brush:", BRUSH_SIZE)

    elif key == ord('s'):
        cv2.imwrite(MASK_OUTPUT, mask)
        print("Saved:", MASK_OUTPUT)

cv2.destroyAllWindows()

Controls
---------------------
Left mouse = draw
c = clear
+ = larger brush
- = smaller brush
s = save mask
ESC = quit
Saved: ../mask_512.png


In [12]:
import cv2

# Load image
image = cv2.imread("../im.png")

# Resize to 512x512
resized = cv2.resize(image, (512, 512), interpolation=cv2.INTER_AREA)

# Save result
cv2.imwrite("../im_512.png", resized)

True

In [17]:
import cv2
import numpy as np

# Load image
image = cv2.imread("../im_512.png")

original = image.copy()

drawing = False
brush_size = 20

def draw(event, x, y, flags, param):
    global drawing, image, brush_size

    if event == cv2.EVENT_LBUTTONDOWN:
        drawing = True

    elif event == cv2.EVENT_MOUSEMOVE:
        if drawing:
            cv2.circle(image, (x, y), brush_size, (0, 0, 255), -1)  # Red in BGR

    elif event == cv2.EVENT_LBUTTONUP:
        drawing = False
        cv2.circle(image, (x, y), brush_size, (0, 0, 255), -1)

cv2.namedWindow("Draw Region")
cv2.setMouseCallback("Draw Region", draw)

while True:
    cv2.imshow("Draw Region", image)

    key = cv2.waitKey(1) & 0xFF

    if key == ord('q'):
        break

    elif key == ord('s'):
        hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
                
        # Red wraps around the HSV hue range, so use two ranges
        lower_red1 = np.array([0, 255, 255])
        upper_red1 = np.array([0, 255, 255])

        lower_red2 = np.array([180, 255, 255])
        upper_red2 = np.array([180, 255, 255])

        # Detect red
        mask_red1 = cv2.inRange(hsv, lower_red1, upper_red1)
        mask_red2 = cv2.inRange(hsv, lower_red2, upper_red2)
        red_mask = cv2.bitwise_or(mask_red1, mask_red2)

        # Keep only the red pixels
        result = cv2.bitwise_and(original, original, mask=red_mask)

        cv2.imwrite("../segmented_im_512.png", result)
        print("Saved as painted_image.png")

    elif key == ord('r'):
        image = original.copy()
        print("Reset.")

    elif key == ord(']'):
        brush_size += 2
        print(f"Brush size: {brush_size}")

    elif key == ord('['):
        brush_size = max(1, brush_size - 2)
        print(f"Brush size: {brush_size}")

cv2.destroyAllWindows()

Saved as painted_image.png


In [24]:
#convert segmented im into a mask
import cv2

img = cv2.imread("../segmented_im_512.png")

gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

_, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY)

cv2.imwrite("../alt_mask_512.png", binary)

True